# BI-кейс: мониторинг заказов интернет-магазина

Учебный проект. Данные созданы синтетически.

**Цель:** какие категории и города дают больше стоимости доставленных заказов, сколько заказов отменяются и как меняются показатели по месяцам?

**План:** 1) загрузить CSV; 2) удалить дубли; 3) создать поля для BI; 4) проверить SQL-запросы; 5) выгрузить CSV для DataLens.


## Шаг 1. Загрузить исходные данные

После запуска ячейки появится кнопка **Choose Files / Выберите файлы**. Выбери `orders_raw.csv` из папки `data` в архиве. По завершении появится таблица с первыми строками.


In [ ]:
import io
import pandas as pd
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
raw = pd.read_csv(io.BytesIO(uploaded[filename]))
print('Строк в исходном файле:', len(raw))
display(raw.head())


**Проверь себя:** должно быть **512 строк** (500 разных заказов и 12 дублей). Если видишь ошибку `FileNotFoundError` или пустую загрузку, запусти ячейку повторно и выбери именно `orders_raw.csv`.


## Шаг 2. Проверить качество и убрать повторяющиеся заказы

`order_id` - уникальный идентификатор заказа. Дубль здесь означает повторно выгруженную строку того же заказа, его нельзя дважды учитывать в метриках.


In [ ]:
print('Повторов по order_id:', raw.duplicated(subset='order_id').sum())
print('Пустые значения по полям:')
print(raw.isna().sum())

df = raw.drop_duplicates(subset='order_id').copy()
print('Уникальных заказов:', len(df))
assert len(df) == 500, 'Ожидается 500 уникальных заказов'


**Ожидаемый ответ:** **12** повторов и **500** уникальных заказов. В `delivery_days` пустые значения ожидаемы, потому что для отменённых и ещё не доставленных заказов срок доставки не определён. Заполнять их нулями неправильно: это испортит среднее время доставки.


## Шаг 3. Создать дополнительные поля для дашборда

- `order_count` - 1 для каждого заказа (чтобы суммировать количество).
- `month` - месяц оформления заказа.
- `is_delivered` - 1, если заказ доставлен, иначе 0.
- `is_finished` - 1, если заказ доставлен или отменён; заказы в обработке не считаем завершёнными.
- `delivered_amount` - стоимость заказа только для доставленных, иначе 0. **Это не прибыль и не выручка Авито.**


In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_count'] = 1
df['month'] = df['order_date'].dt.strftime('%Y-%m')
df['is_delivered'] = (df['status'] == 'delivered').astype(int)
df['is_finished'] = df['status'].isin(['delivered', 'cancelled']).astype(int)
df['delivered_amount'] = df['amount_rub'].where(df['status'] == 'delivered', 0)

print('Количество статусов:')
print(df['status'].value_counts())
print('Стоимость доставленных заказов:', df['delivered_amount'].sum(), 'руб.')
display(df.head())


## Шаг 4. Выполнить SQL в Python без установки базы данных

SQLite - небольшая база данных, доступная прямо в Python. `to_sql` создаёт таблицу `orders`. `GROUP BY` объединяет строки по категории/месяцу, `SUM` складывает значения. После запуска увидишь результат запроса в виде таблицы.


In [ ]:
import sqlite3
connection = sqlite3.connect(':memory:')
df.to_sql('orders', connection, if_exists='replace', index=False)

query = """
SELECT
    category,
    COUNT(*) AS orders_total,
    SUM(is_delivered) AS delivered_orders,
    SUM(delivered_amount) AS delivered_amount_rub
FROM orders
GROUP BY category
ORDER BY delivered_amount_rub DESC;
"""
display(pd.read_sql_query(query, connection))


**Попробуй сама:** поменяй `category` в двух местах SQL (после `SELECT` и после `GROUP BY`) на `city`, затем повторно запусти ячейку. Получишь такую же таблицу, но с группировкой по городам. Это важнее, чем просто скопировать запрос: объясни, почему изменилась группировка.


## Шаг 5. Вторая SQL-задача: доля доставленных среди завершённых

Обрати внимание: если заказ ещё `in_progress`, он **не** должен попадать в знаменатель. Иначе текущий месяц будет выглядеть хуже из-за незавершённых заказов.


In [ ]:
query = """
SELECT
    month,
    SUM(is_delivered) AS delivered_orders,
    SUM(is_finished) AS finished_orders,
    ROUND(100.0 * SUM(is_delivered) / NULLIF(SUM(is_finished), 0), 1) AS delivered_share_pct
FROM orders
GROUP BY month
ORDER BY month;
"""
display(pd.read_sql_query(query, connection))


## Шаг 6. Подготовить CSV для DataLens

Этот файл будет содержать **500 строк** и поля, из которых легко строить графики. После запуска браузер скачает `orders_bi.csv`.


In [ ]:
df['order_date'] = df['order_date'].dt.strftime('%Y-%m-%d')
df.to_csv('orders_bi.csv', index=False, encoding='utf-8-sig')
print('Сохранено строк:', len(df))
files.download('orders_bi.csv')


## Шаг 7. Что сказать на собеседовании

«Я сделала учебный BI-проект на синтетических заказах. Сначала проверила качество выгрузки и убрала 12 дублей по ключу `order_id`. Затем в pandas создала поля для расчёта метрик, SQL использовала для агрегации по месяцам и категориям. В DataLens построила дашборд, где можно посмотреть стоимость доставленных заказов и распределение по категориям и городам. Отдельно оговорила, что `in_progress` не включается в долю успешных завершённых заказов».

**Честная граница:** проект учебный, код создан как тренировочный пример. Обязательно выполни ячейки самостоятельно, измени запрос в шаге 4 и разберись, почему выходит результат, до того как указывать Python и SQL в резюме.
